# ALQAC 2026 — Public Test

Edit only the parameter cell. Run `smoke` first; it automatically performs the model-only gate and then two live end-to-end cases. After it passes, change only `STAGE` to `full` to run 50 cases and evaluate Outcome Accuracy and Law Micro F1. Nothing is uploaded automatically.

In [ ]:
# Edit only this cell.
STAGE = 'smoke'                 # smoke | full
RUN_ID = 'public-candidate-v2'
EXPERIMENT = 'candidate'
GIT_REPO_URL = 'https://github.com/NGBao1608/DL_K23_ALQAC2026.git'
GIT_REF = 'TuanAnh'
APPROVE_PUBLIC_API_CALLS = False  # Set True only after reviewing the irreversible call budget.
ADAPTER_PATH = None

assert STAGE in {'smoke', 'full'}
assert EXPERIMENT in {'baseline', 'candidate'}
if STAGE == 'full' and not APPROVE_PUBLIC_API_CALLS:
    raise ValueError('Public full is live; explicitly approve API calls first.')

## Mount Drive, pin source, and install

This setup cell is intentionally self-contained. `GITHUB_TOKEN` and `ALQAC_TEAM_TOKEN` are required; `HF_TOKEN` is optional.

In [ ]:
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import os
import re
import shutil
import stat
import subprocess
import sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/ALQAC2026')
WORKFLOW_ROOT = DRIVE_ROOT / 'runs/public' / RUN_ID
SOURCE_PIN = WORKFLOW_ROOT / 'source_pin.json'
PROJECT_ROOT = Path('/content/alqac2026')
LOCAL_ROOT = Path('/content/alqac_runtime/public')
for path in (WORKFLOW_ROOT, DRIVE_ROOT / 'cache', DRIVE_ROOT / 'indexes/law', DRIVE_ROOT / 'exports'):
    path.mkdir(parents=True, exist_ok=True)

def secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    value = value.strip() if value and value.strip() else None
    if required and not value:
        raise ValueError(f'Missing or inaccessible Colab Secret: {name}')
    return value

def normalize_git_repo_url(value):
    value = value.strip() if isinstance(value, str) else ''
    markdown = re.fullmatch(r'\[[^\]]+\]\((https://github\.com/[^()\s]+)\)', value)
    if markdown:
        value = markdown.group(1)
    if not re.fullmatch(r'https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?', value):
        raise ValueError('GIT_REPO_URL must be a raw https://github.com/owner/repo.git URL.')
    return value

GIT_REPO_URL = normalize_git_repo_url(GIT_REPO_URL)
github_token = secret('GITHUB_TOKEN')
team_token = secret('ALQAC_TEAM_TOKEN')
hf_token = secret('HF_TOKEN', required=False)
os.chdir(PROJECT_ROOT.parent)
pin = json.loads(SOURCE_PIN.read_text(encoding='utf-8')) if SOURCE_PIN.is_file() else None
if pin is None and STAGE == 'full':
    raise FileNotFoundError('Run smoke first with this RUN_ID to create the source pin.')
if pin and (pin.get('repository') != GIT_REPO_URL or pin.get('branch') != GIT_REF):
    raise ValueError('Existing source pin belongs to a different repository or branch.')

reuse_checkout = False
if pin and (PROJECT_ROOT / '.git').is_dir():
    existing_remote = subprocess.check_output(['git', 'config', '--get', 'remote.origin.url'], cwd=PROJECT_ROOT, text=True).strip()
    existing_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
    existing_dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True).strip()
    reuse_checkout = existing_remote == GIT_REPO_URL and existing_commit == pin.get('commit') and not existing_dirty
if PROJECT_ROOT.exists() and not reuse_checkout:
    shutil.rmtree(PROJECT_ROOT)
if not reuse_checkout:
    askpass = Path('/content/alqac-git-askpass.sh')
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n')
    askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
    git_env = {**os.environ, 'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token}
    try:
        clone = subprocess.run(['git', 'clone', '--branch', GIT_REF, '--single-branch', GIT_REPO_URL, str(PROJECT_ROOT)], env=git_env, text=True, capture_output=True)
        if clone.returncode:
            detail = clone.stderr.strip().splitlines()[-1] if clone.stderr.strip() else 'unknown git error'
            raise RuntimeError(f'Git clone failed for {GIT_REPO_URL}@{GIT_REF}: {detail}')
    finally:
        askpass.unlink(missing_ok=True)
        git_env = None
else:
    print({'checkout_reused': True, 'commit': pin.get('commit')})
github_token = None

if pin:
    pinned_commit = pin.get('commit')
    if not isinstance(pinned_commit, str) or not re.fullmatch(r'[0-9a-fA-F]{40}', pinned_commit):
        raise ValueError('Invalid source pin commit.')
    subprocess.check_call(['git', 'checkout', '--detach', pinned_commit], cwd=PROJECT_ROOT)
else:
    pinned_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
    pin = {'schema_version': 'source-pin-v2', 'repository': GIT_REPO_URL, 'branch': GIT_REF, 'commit': pinned_commit, 'runtime_check_status': 'pending'}
    temp = SOURCE_PIN.with_suffix('.json.tmp')
    temp.write_text(json.dumps(pin, indent=2), encoding='utf-8')
    temp.replace(SOURCE_PIN)

os.chdir(PROJECT_ROOT)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
PIP_BASELINE_PATH = Path('/content/alqac-pip-check-baseline.json')

def pip_check_issues():
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'check'],
        text=True, capture_output=True, check=False,
    )
    output = '\n'.join(
        part.strip() for part in (result.stdout, result.stderr) if part.strip()
    )
    return (set(output.splitlines()) if result.returncode else set()), output

if PIP_BASELINE_PATH.is_file():
    pip_baseline = json.loads(PIP_BASELINE_PATH.read_text(encoding='utf-8'))
else:
    baseline_issues, _ = pip_check_issues()
    pip_baseline = {
        'issues': sorted(baseline_issues),
        'torch_version': importlib.metadata.version('torch'),
    }
    PIP_BASELINE_PATH.write_text(json.dumps(pip_baseline, indent=2), encoding='utf-8')

BOOTSTRAP_STATE_PATH = LOCAL_ROOT / 'bootstrap_state.json'
bootstrap_fingerprint = hashlib.sha256((pinned_commit + '\n' + Path('requirements-colab.txt').read_text(encoding='utf-8') + '\n' + pip_baseline['torch_version']).encode()).hexdigest()
bootstrap_state = json.loads(BOOTSTRAP_STATE_PATH.read_text(encoding='utf-8')) if BOOTSTRAP_STATE_PATH.is_file() else {}
bootstrap_reused = bootstrap_state.get('fingerprint') == bootstrap_fingerprint
if not bootstrap_reused:
    unused = []
    for package in ('gradio', 'gradio-client', 'hf-gradio'):
        try:
            importlib.metadata.version(package)
            unused.append(package)
        except importlib.metadata.PackageNotFoundError:
            pass
    if unused:
        subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', *unused])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'])
else:
    print({'dependency_bootstrap_reused': True})
torch_after = importlib.metadata.version('torch')
if torch_after != pip_baseline['torch_version']:
    raise RuntimeError(
        f"Dependency installation changed Colab torch from {pip_baseline['torch_version']} to {torch_after}. "
        "Restart the runtime."
    )
after_issues, after_output = pip_check_issues()
new_issues = sorted(after_issues - set(pip_baseline['issues']))
if new_issues:
    raise RuntimeError(
        'ALQAC dependency installation introduced new conflicts:\n- '
        + '\n- '.join(new_issues)
    )
if after_issues:
    print('WARNING: Colab has pre-existing pip conflicts; ALQAC introduced none.')
    print(after_output)
else:
    print('pip check: PASS (no broken requirements)')
if not bootstrap_reused:
    BOOTSTRAP_STATE_PATH.write_text(json.dumps({'fingerprint': bootstrap_fingerprint}, indent=2), encoding='utf-8')
print({'torch_preserved': torch_after, 'new_pip_conflicts': len(new_issues)})

project_src = str((PROJECT_ROOT / 'src').resolve())
for module_name in tuple(sys.modules):
    if module_name == 'alqac2026' or module_name.startswith('alqac2026.'):
        del sys.modules[module_name]
sys.path[:] = [entry for entry in sys.path if entry != project_src]
sys.path.insert(0, project_src)
sys.path_importer_cache.pop(project_src, None)
importlib.invalidate_caches()
import alqac2026
import alqac2026.artifacts as artifacts_module
for module in (alqac2026, artifacts_module):
    module_path = Path(module.__file__).resolve()
    if PROJECT_ROOT.resolve() not in module_path.parents:
        raise RuntimeError(f'Stale package import outside pinned checkout: {module_path}')
model_cache_report = artifacts_module.restore_hf_model_snapshots(
    DRIVE_ROOT / 'model_cache', LOCAL_ROOT / 'hf_cache'
)
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
os.environ['HF_HOME'] = str(LOCAL_ROOT / 'hf_cache')
os.environ['TRANSFORMERS_CACHE'] = str(LOCAL_ROOT / 'hf_cache')
print({'stage': STAGE, 'run_id': RUN_ID, 'git_commit': pinned_commit, 'public_api_approved': APPROVE_PUBLIC_API_CALLS, 'model_cache': model_cache_report})

## Run smoke or full

`smoke` performs the zero-call model gate and two live cases. `full` requires the smoke gate, automatically resumes its own checkpoints, runs all 50 cases, evaluates predictions, selects law top-k, and exports a validated candidate.

In [ ]:
from alqac2026.colab_workflow import run_public_stage

if not APPROVE_PUBLIC_API_CALLS:
    raise ValueError('Review the preflight budget, then set APPROVE_PUBLIC_API_CALLS=True.')
os.environ['ALQAC_TEAM_TOKEN'] = team_token
try:
    result = run_public_stage(
        stage=STAGE,
        run_id=RUN_ID,
        repo_root=PROJECT_ROOT,
        drive_root=DRIVE_ROOT,
        local_root=LOCAL_ROOT,
        config_path=PROJECT_ROOT / f'configs/{EXPERIMENT}.yaml',
        approved_live_api=APPROVE_PUBLIC_API_CALLS,
        adapter_path=ADAPTER_PATH,
    )
finally:
    team_token = None
    os.environ.pop('ALQAC_TEAM_TOKEN', None)

print({'status': result['validation']['status'], 'stage': STAGE, 'run_dir': result['run_dir'], 'api_plan': result['api_plan']})

## Public evaluation

This cell is meaningful after `full`. It does not estimate official Penalized Case Recall when Public gold opaque segment IDs are unavailable.

In [ ]:
if STAGE == 'full':
    full_dir = WORKFLOW_ROOT / 'full'
    metrics = json.loads((full_dir / 'metrics.json').read_text(encoding='utf-8'))
    selection = json.loads((full_dir / 'selection_profile.json').read_text(encoding='utf-8'))
    errors = json.loads((full_dir / 'errors.json').read_text(encoding='utf-8'))
    print({
        'outcome_accuracy': metrics['outcome_accuracy'],
        'law_micro_f1': metrics['law_micro_f1'],
        'law_recall_at_5': metrics['law_recall_at_5'],
        'format_failures': metrics['format_failures'],
        'submission_law_top_k': selection['submission_law_top_k'],
        'error_cases': len(errors),
        'export_dir': result['export_dir'],
    })
else:
    print('Smoke PASS. Change only STAGE to full and Run all again with the same RUN_ID.')